# Lab 9.7 &mdash; Validate Grounding

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Check every claim maps to a real figure in the report
- Check each claim cites the correct source
- Refuse to ship a summary with an ungrounded claim

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; grounding/citation/compute logic and, in the agent-assembly labs, tool wiring &amp; the read-only guardrail &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; Grounding: RAG & citations](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-09-07"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Never ship an ungrounded claim (deck slides 4, 8, 14). Before the summary goes to an analyst, the agent
**validates** it: every claim must map to a **real figure** in the report, and it must cite the
**correct source**. A claim that cites the wrong page, or a figure that isn't in the report, is a
grounding failure &mdash; don't ship it.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

print("a claim must match REPORT[metric] on BOTH value-source and source string")

## Your Turn
Complete `validate_summary`: collect an ungrounded claim and a wrong-source claim.

In [ ]:
def validate_summary(claims, report):
    problems = []
    for c in claims:
        fig = report.get(c['metric'])
        if fig is None:
            problems.append("ungrounded: " + c["metric"])   # TODO: keep this line
        elif ___:   # TODO: the claim cites a source that does not match the report
            problems.append("wrong source: " + c["metric"])
    return {"ok": ___, "problems": problems}   # TODO: ok = no problems

try:
    good = [{'metric': 'revenue', 'source': 'p4, income stmt'}]
    ungrounded = [{'metric': 'ebitda', 'source': 'p9'}]
    wrong = [{'metric': 'revenue', 'source': 'p1, cover'}]
    print('good      ->', validate_summary(good, REPORT))
    print('ungrounded->', validate_summary(ungrounded, REPORT))
    print('wrong src ->', validate_summary(wrong, REPORT))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a correctly-grounded summary passes", lambda: validate_summary([{"metric": "revenue", "source": "p4, income stmt"}], REPORT)["ok"] is True)
expect_true("a claim about a missing figure is caught", lambda: any("ungrounded" in p for p in validate_summary([{"metric": "ebitda", "source": "p9"}], REPORT)["problems"]))
expect_true("a wrong-source citation is caught", lambda: any("wrong source" in p for p in validate_summary([{"metric": "revenue", "source": "p1, cover"}], REPORT)["problems"]))
expect_true("ok is False when any problem exists", lambda: validate_summary([{"metric": "ebitda", "source": "p9"}], REPORT)["ok"] is False)
expect_true("multiple problems are all collected", lambda: len(validate_summary([{"metric": "ebitda", "source": "p9"}, {"metric": "revenue", "source": "wrong"}], REPORT)["problems"]) == 2)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Validation is the gate before an analyst sees the summary: every claim must map to a real figure and cite the right source. An ungrounded or mis-cited claim doesn't ship -- that's the high-stakes bar.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>